In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import torch.optim as optim
import torchvision.datasets as datasets
from torchvision import models, transforms
from torchvision.utils import save_image, make_grid
from torch.optim.lr_scheduler import StepLR
from torch import autograd
from torch.autograd import Variable
from tensorboardX import SummaryWriter
from torch.utils.data import Dataset
from typing import Dict, Tuple
from tqdm import tqdm
import numpy as np
import time
import os
import random
from tabulate import tabulate
import h5py
import wandb

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

%matplotlib inline

if __name__ == "__main__":
    print("Torch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    print("Number of GPUs:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else "No GPU detected")

Torch version: 2.9.0+cu128
CUDA available: True
CUDA version: 12.8
Number of GPUs: 1
GPU name: NVIDIA GeForce RTX 3090


In [2]:
import os, sys, platform, pathlib
print("platform:", sys.platform, os.name, platform.system())   # expect: linux posix Linux
print("cwd     :", os.getcwd())                                # should start with /home/...
print("python  :", sys.executable)                             # should be in /usr/bin/python or /home/.../.venv/...
print("repo path looks unix? ", os.getcwd().startswith("/home/"))


platform: linux posix Linux
cwd     : /home/omiqran/projects/metamaterials_urop
python  : /home/omiqran/projects/metamaterials_urop/.venv/bin/python
repo path looks unix?  True


In [3]:
# Warm just the slices you actually use (fast and enough to prime cache)
import h5py, numpy as np, time
h5 = "/home/omiqran/projects/metamaterials_urop/train_test_split.h5"

t0 = time.time()
with h5py.File(h5, "r") as f:
    d = f["pattern_train"]
    n = d.shape[0]
    bs = 4096  # contiguous blocks; tune
    for i in range(0, min(n, 8*bs), bs):   # warm ~8 blocks; adjust as desired
        _ = d[i:i+bs, :32, :32]            # exactly what the loader reads
    _ = f["params_train"][:10000]          # light touch of params
    _ = f["neff_train"][:10000, 0:1]       # light touch of neff[:,0]
print("warmup took", time.time()-t0, "s")


warmup took 2.350181818008423 s


In [ ]:
#  # Load data
# BATCH_SIZE = 512
# filename = r"train_test_split.h5"

# def quarter(pattern):
#     return pattern[:32, :32]  # downsample to 32x32

# # Custom Dataset class
# class NeffDataset(Dataset):
#     def __init__(self, patterns, params, targets):
#         self.patterns = torch.FloatTensor(patterns)
#         self.params = torch.FloatTensor(params)
#         self.targets = torch.FloatTensor(targets)
    
#     def __len__(self):
#         return len(self.patterns)
    
#     def __getitem__(self, idx):
#         return self.patterns[idx], self.params[idx], self.targets[idx]
    
# def load_data(filename):
#     """Load and process data from HDF5 file"""
#     with h5py.File(filename, "r") as f:
#         neff_train = np.array(f['neff_train'])
#         weight_train = np.array(f['weight_train'])
#         params_train = np.array(f['params_train'])
#         pattern_train = np.array(f['pattern_train'])  # shape [N, 64, 64]
        
#         neff_test = np.array(f['neff_test'])
#         weight_test = np.array(f['weight_test'])
#         params_test = np.array(f['params_test'])
#         pattern_test = np.array(f['pattern_test'])  # shape [M, 64, 64]
    
#     # Apply quarter() to every pattern (vectorized alternative)
#     pattern_train_quartered = pattern_train[:, :32, :32]  # shape [N, 32, 32]
#     pattern_test_quartered  = pattern_test[:, :32, :32]   # shape [M, 32, 32]

#     # Reshape for downstream use if needed
#     xs  = pattern_train_quartered[:, np.newaxis, :, :]
#     xs1 = pattern_test_quartered[:,  np.newaxis, :, :]

#     x1s  = params_train
#     x1s1 = params_test
#     ys   = neff_train[:, 0:1]
#     ys1  = neff_test[:, 0:1]
    
#     return xs, x1s, ys, xs1, x1s1, ys1

# def create_dataloaders(xs, x1s, ys, xs1, x1s1, ys1, batch_size=256):
#     """Create training and test dataloaders with Windows-compatible settings"""
#     train_dataset = NeffDataset(xs, x1s, ys)
#     test_dataset = NeffDataset(xs1, x1s1, ys1)
    
#     # Use single process for Windows compatibility
#     train_loader = DataLoader(
#         train_dataset, 
#         batch_size=batch_size, 
#         shuffle=False,
#         num_workers=0,  # Use 0 for Windows compatibility
#         pin_memory=True
#     )
#     test_loader = DataLoader(
#         test_dataset, 
#         batch_size=batch_size, 
#         shuffle=False,
#         num_workers=0,  # Use 0 for Windows compatibility
#         pin_memory=True
#     )
    
#     return train_loader, test_loader


# try:
#     xs, x1s, ys, xs1, x1s1, ys1 = load_data(filename)
#     print(f"Training data shape: {xs.shape}")
#     print(f"Training params shape: {x1s.shape}")
#     print(f"Training targets shape: {ys.shape}")
#     print(f"Test data shape: {xs1.shape}")
#     print(f"Test params shape: {x1s1.shape}")
#     print(f"Test targets shape: {ys1.shape}")
# except FileNotFoundError:
#     print(f"Data file not found: {filename}")
#     print("Please update the filename path")
#     exit()

# # Create dataloaders
# train_loader, test_loader = create_dataloaders(xs, x1s, ys, xs1, x1s1, ys1, batch_size=BATCH_SIZE)
# print(f"Number of training batches: {len(train_loader)}")
# print(f"Number of test batches: {len(test_loader)}")

In [5]:
import os
import h5py
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# # -------------------------
# # Config
# # -------------------------
BATCH_SIZE = 512
filename = "/home/omiqran/projects/metamaterials_urop/train_test_split.h5"
  # move this file under /home/... (Linux FS), not /mnt/c/...

# # -------------------------
# # Lazy HDF5 Dataset (WSL-safe)
# # -------------------------
# class H5QuarterDataset(Dataset):
#     """
#     Lazily reads items from the HDF5 file.
#     - Returns:
#         x_img:  (1, 32, 32)  float32
#         x_par:  (4,)         float32
#         y:      (1,)         float32  (neff[:, 0:1])
#     """
#     def __init__(self, h5_path: str, split: str = "train"):
#         assert split in ("train", "test")
#         self.h5_path = h5_path
#         self.split = split
#         self._f = None  # per-process file handle

#         # read sizes without keeping the file open
#         with h5py.File(self.h5_path, "r") as f:
#             self._pattern_shape = f[f"pattern_{split}"].shape   # (N, 64, 64)
#             self._params_shape  = f[f"params_{split}"].shape    # (N, 4)
#             self._neff_shape    = f[f"neff_{split}"].shape      # (N, M)
#             self.N = self._pattern_shape[0]

#     def _ensure_open(self):
#         if self._f is None:
#             self._f = h5py.File(self.h5_path, "r")
#             self._pattern = self._f[f"pattern_{self.split}"]
#             self._params  = self._f[f"params_{self.split}"]
#             self._neff    = self._f[f"neff_{self.split}"]

#     def __len__(self):
#         return self.N

#     def __getitem__(self, idx):
#         self._ensure_open()
#         # HDF5 -> numpy (lazy slice)
#         pat64 = self._pattern[idx]          # (64,64), dtype whatever in file
#         pat32 = pat64[:32, :32]             # quarter to (32,32)

#         x_img = torch.from_numpy(pat32).unsqueeze(0).float()      # (1,32,32)
#         x_par = torch.from_numpy(self._params[idx]).float()       # (4,)
#         y     = torch.from_numpy(self._neff[idx, 0:1]).float()    # (1,)
#         return x_img, x_par, y

# # -------------------------
# # Utility: report shapes without loading everything
# # -------------------------
# def report_shapes(h5_path: str):
#     with h5py.File(h5_path, "r") as f:
#         N_tr = f["pattern_train"].shape[0]
#         N_te = f["pattern_test"].shape[0]

#         # After quartering and adding channel dim:
#         tr_img_shape = (N_tr, 1, 32, 32)
#         te_img_shape = (N_te, 1, 32, 32)

#         # Params assumed (N,4); Targets (N,1) from neff[:,0:1]
#         tr_par_shape = (N_tr, f["params_train"].shape[1])
#         te_par_shape = (N_te, f["params_test"].shape[1])
#         tr_y_shape   = (N_tr, 1)
#         te_y_shape   = (N_te, 1)

#     print(f"Training data shape: {tr_img_shape}")
#     print(f"Training params shape: {tr_par_shape}")
#     print(f"Training targets shape: {tr_y_shape}")
#     print(f"Test data shape: {te_img_shape}")
#     print(f"Test params shape: {te_par_shape}")
#     print(f"Test targets shape: {te_y_shape}")

# # -------------------------
# # Build datasets/loaders (WSL-friendly defaults)
# # -------------------------
# def build_loaders(h5_path: str, batch_size: int = 512):
#     train_ds = H5QuarterDataset(h5_path, split="train")
#     test_ds  = H5QuarterDataset(h5_path, split="test")

#     # num_workers=0 avoids h5py/fork issues; raise to 1–2 later with spawn + persistent_workers
#     train_loader = DataLoader(
#         train_ds,
#         batch_size=batch_size,
#         shuffle=False,          # keep behavior identical to your original
#         num_workers=0,
#         persistent_workers=False,
#         pin_memory=False        # turn on later only if you push to GPU and see benefit
#     )
#     test_loader = DataLoader(
#         test_ds,
#         batch_size=batch_size,
#         shuffle=False,
#         num_workers=0,
#         persistent_workers=False,
#         pin_memory=False
#     )
#     return train_loader, test_loader

# # -------------------------
# # Main
# # -------------------------
# if __name__ == "__main__":
#     if not os.path.exists(filename):
#         print(f"Data file not found: {filename}")
#         raise SystemExit(1)

#     # Print shapes (matches your previous printout, but without full RAM load)
#     report_shapes(filename)

#     train_loader, test_loader = build_loaders(filename, batch_size=BATCH_SIZE)

#     # Batch counts should match your originals: 2176 and 933 with BATCH_SIZE=512
#     print(f"Number of training batches: {len(train_loader)}")
#     print(f"Number of test batches: {len(test_loader)}")

#     # Optional quick sanity read (first batch only)
#     xb, xp, y = next(iter(train_loader))
#     # xb: (B,1,32,32), xp: (B,4), y: (B,1)
#     # print(xb.shape, xp.shape, y.shape)


In [6]:
# import math
# import numpy as np
# import h5py
# import torch
# from torch.utils.data import IterableDataset, DataLoader, get_worker_info

# def _open_h5(h5_path: str):
#     return h5py.File(
#         h5_path, "r",
#         libver="latest", swmr=True,
#         rdcc_nslots=1_000_003, rdcc_nbytes=64 * 1024 * 1024, rdcc_w0=0.75
#     )

# class H5QuarterBatchIterable(IterableDataset):
#     """
#     IterableDataset that yields **batches** directly (B,1,32,32), (B,4), (B,1)
#     so we do one HDF5 slice per batch. Much faster when I/O-bound.
#     """
#     def __init__(self, h5_path: str, split: str = "train", batch_size: int = 512):
#         assert split in ("train", "test")
#         self.h5_path = h5_path
#         self.split = split
#         self.batch_size = batch_size
#         with h5py.File(self.h5_path, "r") as f:
#             self.N = f[f"pattern_{split}"].shape[0]
#     def __len__(self):
#         # Approximate total number of batches for single-process iteration.
#         # If you use num_workers>0, this is the global length, not per-worker.
#         import math
#         return math.ceil(self.N / self.batch_size)

#     def _open(self):
#         f = _open_h5(self.h5_path)
#         s = self.split
#         return f, f[f"pattern_{s}"], f[f"params_{s}"], f[f"neff_{s}"]

#     def __iter__(self):
#         # Split index range across workers
#         info = get_worker_info()
#         if info is None:
#             start, end = 0, self.N
#         else:
#             per_worker = int(math.ceil(self.N / info.num_workers))
#             start = info.id * per_worker
#             end = min(start + per_worker, self.N)

#         f, pat, par, neff = self._open()

#         B = self.batch_size
#         for lo in range(start, end, B):
#             hi = min(lo + B, end)

#             # Vectorized reads
#             p = pat[lo:hi, :32, :32]  # (b,32,32)
#             if p.dtype != np.float32:
#                 p = p.astype(np.float32, copy=False)
#             x_img = torch.from_numpy(p).unsqueeze(1)    # (b,1,32,32)

#             pr = par[lo:hi]                             # (b,4)
#             if pr.dtype != np.float32:
#                 pr = pr.astype(np.float32, copy=False)
#             x_par = torch.from_numpy(pr)                # (b,4)

#             yv = neff[lo:hi, 0:1]                       # (b,1)
#             if yv.dtype != np.float32:
#                 yv = yv.astype(np.float32, copy=False)
#             y = torch.from_numpy(yv)                    # (b,1)

#             yield x_img, x_par, y

#         f.close()  # cleanup per worker

# def build_loaders_fast_iterable(h5_path: str, batch_size: int = 512, num_workers: int = 0):
#     train_ds = H5QuarterBatchIterable(h5_path, "train", batch_size)
#     test_ds  = H5QuarterBatchIterable(h5_path, "test",  batch_size)

#     train_loader = DataLoader(
#         train_ds,
#         batch_size=None,             # IterableDataset already yields batches
#         num_workers=num_workers,     # 0 on Windows/WSL; >0 on Linux for speed
#         persistent_workers=bool(num_workers),
#         pin_memory=False
#     )
#     test_loader = DataLoader(
#         test_ds,
#         batch_size=None,
#         num_workers=num_workers,
#         persistent_workers=bool(num_workers),
#         pin_memory=False
#     )
#     return train_loader, test_loader

import os
import h5py
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, get_worker_info

BATCH_SIZE = 512
# filename = r"train_test_split.h5"   # keep this on Linux FS (e.g., /home/…), not /mnt/c

def _open_h5(h5_path: str):
    # Per-process/worker handle; SWMR + larger raw-data chunk cache for faster random reads
    # rdcc_nbytes ~ 64MB; tune if you have more RAM
    return h5py.File(
        h5_path, "r",
        libver="latest", swmr=True,
        rdcc_nslots=1_000_003, rdcc_nbytes=64 * 1024 * 1024, rdcc_w0=0.75
    )

class H5QuarterDatasetFast(Dataset):
    """
    Map-style dataset that keeps a per-process/worker HDF5 handle open.
    Returns: x_img (1,32,32) float32, x_par (4,), y (1,)
    """
    def __init__(self, h5_path: str, split: str = "train"):
        assert split in ("train", "test")
        self.h5_path = h5_path
        self.split = split
        self._f = None  # per-process/worker handle
        # sizes without keeping the file open
        with h5py.File(self.h5_path, "r") as f:
            self.N = f[f"pattern_{split}"].shape[0]

    def _file(self):
        if self._f is None:
            self._f = _open_h5(self.h5_path)
            s = self.split
            self._pattern = self._f[f"pattern_{s}"]  # (N,64,64)
            self._params  = self._f[f"params_{s}"]   # (N,4)
            self._neff    = self._f[f"neff_{s}"]     # (N,M)
        return self._f

    def __len__(self):
        return self.N

    def __getitem__(self, idx):
        self._file()  # ensure open
        # ---- vector reads (single index) ----
        p64 = self._pattern[idx]              # np array
        p32 = p64[:32, :32]
        # Zero-copy to tensor when already float32; else cheap astype
        if p32.dtype != np.float32:
            p32 = p32.astype(np.float32, copy=False)
        x_img = torch.from_numpy(p32).unsqueeze(0)     # (1,32,32)

        par = self._params[idx]
        if par.dtype != np.float32:
            par = par.astype(np.float32, copy=False)
        x_par = torch.from_numpy(par)                  # (4,)

        yv = self._neff[idx, 0:1]
        if yv.dtype != np.float32:
            yv = yv.astype(np.float32, copy=False)
        y = torch.from_numpy(yv)                       # (1,)
        return x_img, x_par, y

def _worker_init_fn(_):
    # Each worker gets its own handle (avoid sharing h5py File across processes)
    info = get_worker_info()
    if info is not None and hasattr(info.dataset, "_f"):
        info.dataset._f = None

def build_loaders_fast_map(h5_path: str, batch_size: int = 512, num_workers: int = 0):
    train_ds = H5QuarterDatasetFast(h5_path, "train")
    test_ds  = H5QuarterDatasetFast(h5_path, "test")

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,                 # start with 0 on Windows/WSL
        worker_init_fn=_worker_init_fn if num_workers > 0 else None,
        persistent_workers=bool(num_workers),
        pin_memory=False
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        worker_init_fn=_worker_init_fn if num_workers > 0 else None,
        persistent_workers=bool(num_workers),
        pin_memory=False
    )
    return train_loader, test_loader



# if __name__ == "__main__":
#     if not os.path.exists(filename):
#         print(f"Data file not found: {filename}")
#         raise SystemExit(1)

#     # Print shapes (matches your previous printout, but without full RAM load)
#     # report_shapes(filename)

#     train_loader, test_loader = build_loaders(filename, batch_size=BATCH_SIZE)

#     # Batch counts should match your originals: 2176 and 933 with BATCH_SIZE=512
#     print(f"Number of training batches: {len(train_loader)}")
#     print(f"Number of test batches: {len(test_loader)}")

#     # Optional quick sanity read (first batch only)
#     xb, xp, y = next(iter(train_loader))
#     # xb: (B,1,32,32), xp: (B,4), y: (B,1)
#     # print(xb.shape, xp.shape, y.shape)

if __name__ == "__main__":
    assert os.path.exists(filename), f"Missing file: {filename}"

    # Choose ONE:
    train_loader, test_loader = build_loaders_fast_map(filename, batch_size=BATCH_SIZE, num_workers=0)
    # train_loader, test_loader = build_loaders_fast_iterable(filename, batch_size=BATCH_SIZE, num_workers=0)

    print(f"Train batches: {len(train_loader) if hasattr(train_loader, '__len__') else 'iterable'}")
    xb, xp, y = next(iter(train_loader))
    print(xb.shape, xp.shape, y.shape)  # (B,1,32,32) (B,4) (B,1)


Train batches: 2176
torch.Size([512, 1, 32, 32]) torch.Size([512, 4]) torch.Size([512, 1])


In [7]:
for i, (a,b,c) in enumerate(train_loader):
    print(i)
    print(a,b,c)
    break

0
tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]],


        [[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]],


        [[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]],


        ...,


        [[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 

In [8]:
import math
from typing import List, Optional, Callable, Sequence, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F


def _safe_groupnorm_groups(ch: int, preferred: int = 8) -> int:
    """
    Choose a number of groups for GroupNorm that divides 'ch'.
    Tries 'preferred', then decrements until it finds a divisor, falling back to 1.
    """
    g = min(preferred, ch)
    while g > 1 and (ch % g != 0):
        g -= 1
    return g


class ConvBlock(nn.Module):
    def __init__(
        self,
        in_ch: int,
        out_ch: int,
        *,
        act: Optional[Callable[[], nn.Module]] = None,
        dropout_p: float = 0.0,
        use_dropout: bool = False,
        norm_groups_preferred: int = 8,
        kernel_size: int = 3,
        padding: int = 1,
        stride: int = 1,
    ):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=kernel_size, padding=padding, stride=stride)
        g = _safe_groupnorm_groups(out_ch, norm_groups_preferred)
        self.norm = nn.GroupNorm(g, out_ch)
        self.act  = act() if act is not None else nn.GELU()
        self.drop = nn.Dropout(dropout_p) if use_dropout and dropout_p > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        x = self.norm(x)
        x = self.act(x)
        x = self.drop(x)
        return x


class ModularCondCNN(nn.Module):
    """
    A modular CNN that ingests an image (B,1,H,W) and a conditioning vector (B,cond_dim),
    tiles the conditioning to the current spatial size, concatenates it, and proceeds deeper.

    Configure depth/width via lists:
      - pre_concat_blocks:  List[Tuple[out_channels, pool_after(bool)]]
      - post_concat_blocks: List[out_channels]  (no pooling here; add if you want)
      - head_dims:          List[int] for MLP layers before the final scalar output

    Dropout:
      - conv_dropout_p, fc_dropout_p
      - conv_drop_every: apply dropout to every k-th conv block (1 = all, 2 = every other, etc.)

    GroupNorm:
      - norm_groups_preferred: target number of groups (auto-adjusted to divide channels)

    Conditioning:
      - cond_dim: dimension of x_cond (defaults to 4). Tiled to match feature spatial size.
    """

    def __init__(
        self,
        *,
        in_channels: int = 1,
        cond_dim: int = 4,
        pre_concat_blocks: Sequence[Tuple[int, bool]] = ((64, True), (128, True), (256, False)),
        post_concat_blocks: Sequence[int] = (256, 256, 256, 256, 256),
        head_dims: Sequence[int] = (512, 128),
        act_factory: Callable[[], nn.Module] = nn.GELU,
        conv_dropout_p: float = 0.22,
        fc_dropout_p: float = 0.22,
        conv_drop_every: int = 2,  # e.g. 2 = every other (like your original)
        norm_groups_preferred: int = 8,
        kernel_size: int = 3,
        padding: int = 1,
        pool_kernel: int = 2,
        pool_stride: int = 2,
    ):
        super().__init__()

        self.cond_dim = cond_dim
        self.pool = nn.MaxPool2d(pool_kernel, pool_stride)
        self.act_factory = act_factory
        self.conv_dropout_p = conv_dropout_p
        self.fc_dropout_p = fc_dropout_p
        self.conv_drop_every = max(1, conv_drop_every)
        self.norm_groups_preferred = norm_groups_preferred
        self.kernel_size = kernel_size
        self.padding = padding

        # ---- Build pre-concat conv stack (optionally pooling after each block) ----
        convs_pre = []
        in_ch = in_channels
        block_idx = 1
        for out_ch, pool_after in pre_concat_blocks:
            use_dropout = (block_idx % self.conv_drop_every == 0)
            convs_pre.append(
                ConvBlock(
                    in_ch, out_ch,
                    act=act_factory,
                    dropout_p=conv_dropout_p,
                    use_dropout=use_dropout,
                    norm_groups_preferred=norm_groups_preferred,
                    kernel_size=kernel_size,
                    padding=padding,
                )
            )
            if pool_after:
                convs_pre.append(nn.MaxPool2d(pool_kernel, pool_stride))
            in_ch = out_ch
            block_idx += 1
        self.pre = nn.Sequential(*convs_pre)

        # ---- After pre, we will concat tiled conditioning: channels += cond_dim ----
        in_ch_after = in_ch + cond_dim

        # ---- Build post-concat conv stack (no pooling by default) ----
        convs_post = []
        for out_ch in post_concat_blocks:
            use_dropout = (block_idx % self.conv_drop_every == 0)
            convs_post.append(
                ConvBlock(
                    in_ch_after, out_ch,
                    act=act_factory,
                    dropout_p=conv_dropout_p,
                    use_dropout=use_dropout,
                    norm_groups_preferred=norm_groups_preferred,
                    kernel_size=kernel_size,
                    padding=padding,
                )
            )
            in_ch_after = out_ch
            block_idx += 1
        self.post = nn.Sequential(*convs_post)

        # ---- Head: infer flatten size lazily, so we need a small probe in forward ----
        # We'll create the linear layers on first forward pass when we know spatial dims.
        self.head_dims = list(head_dims)
        self.fc_layers: Optional[nn.Sequential] = None
        self.final: Optional[nn.Linear] = None

        self.fc_dropout = nn.Dropout(fc_dropout_p) if fc_dropout_p > 0 else nn.Identity()

# Inside class ModularCondCNN
    def _build_head(self, feat: torch.Tensor):
        b, c, h, w = feat.shape
        flat = c * h * w
        layers = []
        in_dim = flat
        for hd in self.head_dims:
            layers.append(nn.Linear(in_dim, hd))
            layers.append(self.act_factory())
            if isinstance(self.fc_dropout, nn.Dropout):
                layers.append(self.fc_dropout)
            in_dim = hd
        self.fc_layers = nn.Sequential(*layers) if layers else nn.Identity()
        self.final = nn.Linear(in_dim, 1)

        # >>> NEW: ensure correct device/dtype
        dev, dt = feat.device, feat.dtype
        self.fc_layers.to(device=dev, dtype=dt)
        self.final.to(device=dev, dtype=dt)


    def forward(self, x_img: torch.Tensor, x_cond: torch.Tensor) -> torch.Tensor:
        """
        x_img:  [B, 1, H, W]    (H=W=32 in your current setup, but not strictly required)
        x_cond: [B, cond_dim]
        """
        # ---- Pre-concat stack ----
        x = self.pre(x_img)

        # ---- Tile conditioning to current spatial size ----
        b, _, h, w = x.shape
        cond = x_cond.view(b, self.cond_dim, 1, 1).expand(-1, -1, h, w)
        x = torch.cat([x, cond], dim=1)

        # ---- Post-concat stack ----
        x = self.post(x)

        # ---- Head (lazy build) ----
        if self.fc_layers is None or self.final is None:
            self._build_head(x)

        x = torch.flatten(x, 1)
        x = self.fc_layers(x)
        out = self.final(x)
        return out


# ---------------------------
# Example: match your original
# ---------------------------
def example_cfg_original() -> ModularCondCNN:
    """
    This config mirrors your original architecture:
      Pre:
        - Conv(1->64) + Pool
        - Conv(64->128) + Pool
        - Conv(128->256) (no pool)
      Concat cond_dim=4 --> channels + 4 (256+4=260)
      Post:
        - 5 × Conv(256)
      Head:
        - [512, 128] -> 1
      Dropout:
        - conv: every other block (like your code), p=0.22
        - fc: p=0.22
    """
    return ModularCondCNN(
        in_channels=1,
        cond_dim=4,
        pre_concat_blocks=((64, True), (128, True), (256, False)),
        post_concat_blocks=(256, 256, 256, 256, 256),
        head_dims=(512, 128),
        act_factory=nn.GELU,
        conv_dropout_p=0.22,
        fc_dropout_p=0.22,
        conv_drop_every=2,
        norm_groups_preferred=8,
        kernel_size=3,
        padding=1,
        pool_kernel=2,
        pool_stride=2,
    )


# ---------------------------
# Example: wider & deeper
# ---------------------------
def example_cfg_wide_deep() -> ModularCondCNN:
    """
    A beefier variant for sweeps. Still ends at ~8×8 if you keep two pools early.
    """
    return ModularCondCNN(
        in_channels=1,
        cond_dim=4,
        pre_concat_blocks=((96, True), (192, True), (320, False)),
        post_concat_blocks=(320, 320, 320, 320, 320, 320),
        head_dims=(768, 192),
        conv_dropout_p=0.2,
        fc_dropout_p=0.3,
        conv_drop_every=2,
    )

In [9]:
# train_modular_condcnn_sweep.py
import os
import math
import random
from typing import List, Tuple, Sequence, Optional

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset, SubsetRandomSampler

import wandb
import matplotlib.pyplot as plt

# If your ModularCondCNN lives in another file, import it:
# from your_module import ModularCondCNN

# --------------------------
# Utilities
# --------------------------
def seed_everything(seed: int = 1337):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_subset_loaders_from_existing_train_loader(
    base_train_loader: DataLoader,
    train_frac: float = 0.1,
    val_frac: float = 0.1,
    batch_size: Optional[int] = None,
    seed: int = 1337,
) -> Tuple[DataLoader, DataLoader]:
    """
    Split the dataset behind `train_loader` into small train/val subsets.
    Keeps your base collate_fn & num_workers; uses a new batch_size if provided.
    """
    assert 0 < train_frac < 1 and 0 < val_frac < 1 and train_frac + val_frac < 1.0

    ds = base_train_loader.dataset
    n = len(ds)
    n_train = max(1, int(n * train_frac))
    n_val   = max(1, int(n * val_frac))
    n_rest  = n - (n_train + n_val)
    if n_rest < 0:
        n_val = max(1, n - n_train)

    # Reproducible split
    g = torch.Generator()
    g.manual_seed(seed)
    subsets = random_split(ds, [n_train, n_val, n - (n_train + n_val)], generator=g)
    ds_train, ds_val = subsets[0], subsets[1]

    base_bs = base_train_loader.batch_size or 32
    bs = batch_size or base_bs

    train_loader_small = DataLoader(
        ds_train,
        batch_size=bs,
        shuffle=True,
        num_workers=base_train_loader.num_workers,
        pin_memory=getattr(base_train_loader, "pin_memory", False),
        collate_fn=base_train_loader.collate_fn,
        drop_last=False,
    )

    val_loader_small = DataLoader(
        ds_val,
        batch_size=bs,
        shuffle=False,
        num_workers=base_train_loader.num_workers,
        pin_memory=getattr(base_train_loader, "pin_memory", False),
        collate_fn=base_train_loader.collate_fn,
        drop_last=False,
    )

    return train_loader_small, val_loader_small


def to_device(batch, device):
    # Expecting a batch like (x_img, x_cond, y)
    x_img, x_cond, y = batch
    return x_img.to(device), x_cond.to(device), y.to(device)


def build_model_from_config(config) -> nn.Module:
    """
    Translate W&B config into a ModularCondCNN instance.
    We sweep most knobs with 3 values (small/mid/big).
    """
    # Activation
    act_map = {
        "gelu": nn.GELU,
        "relu": nn.ReLU,
        "silu": nn.SiLU,
    }
    act_factory = act_map[config.act]

    # Pre-concat blocks (tuples: (out_channels, pool_after))
    pre_options = {
        "small": [(32, True), (64, True), (128, False)],
        "mid":   [(64, True), (128, True), (256, False)],   # your original scale
        "big":   [(96, True), (192, True), (320, False)],
    }
    pre_concat_blocks = pre_options[config.pre_size]

    # Post-concat stack: (channels repeated, depth varies)
    post_options = {
        "small": [192, 192, 192],
        "mid":   [256, 256, 256, 256, 256],
        "big":   [320, 320, 320, 320, 320, 320],
    }
    post_concat_blocks = post_options[config.post_size]

    # Head MLP
    head_map = {
        "none":   [],
        "mid":    [512, 128],
        "wide":   [768, 192],
    }
    head_dims = head_map[config.head]

    # Kernel / padding (keep padding=same-ish)
    kernel_size = int(config.kernel_size)
    padding = kernel_size // 2

    # Pool (stride = kernel for simplicity)
    pool_kernel = int(config.pool_kernel)
    pool_stride = pool_kernel

    model = ModularCondCNN(
        in_channels=1,
        cond_dim=4,  # Your conditioning dimension
        pre_concat_blocks=pre_concat_blocks,
        post_concat_blocks=post_concat_blocks,
        head_dims=head_dims,
        act_factory=act_factory,
        conv_dropout_p=float(config.conv_dropout),
        fc_dropout_p=float(config.fc_dropout),
        conv_drop_every=int(config.conv_drop_every),
        norm_groups_preferred=int(config.norm_groups),
        kernel_size=kernel_size,
        padding=padding,
        pool_kernel=pool_kernel,
        pool_stride=pool_stride,
    )
    return model


def plot_and_log_curves(train_losses: List[float], val_losses: List[float], out_path: str = "loss_curve.png"):
    plt.figure()
    plt.plot(range(1, len(train_losses) + 1), train_losses, label="Train MSE")
    plt.plot(range(1, len(val_losses) + 1), val_losses, label="Val MSE")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    wandb.log({"loss_curve": wandb.Image(out_path)})


# --------------------------
# Training loop
# --------------------------
def train_one_run(config=None):
    with wandb.init(config=config):
        cfg = wandb.config

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        seed_everything(cfg.seed)

        # ---- Use your existing train_loader (imported from your code) ----
        # It must be available in this namespace.
        global train_loader  # ensure we refer to the user's provided loader

        # Small subset split for speed
        train_loader_small, val_loader_small = make_subset_loaders_from_existing_train_loader(
            train_loader,
            train_frac=cfg.train_frac,
            val_frac=cfg.val_frac,
            batch_size=cfg.batch_size,
            seed=cfg.seed,
        )

        # Build model
        model = build_model_from_config(cfg).to(device)

        # Optimizer
        opt_name = cfg.optimizer
        if opt_name == "adam":
            optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        elif opt_name == "adamw":
            optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        else:  # "sgd"
            optimizer = optim.SGD(model.parameters(), lr=cfg.lr, momentum=0.9, weight_decay=cfg.weight_decay)

        scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp)
        loss_fn = nn.MSELoss()

        epochs = int(cfg.epochs)
        train_losses, val_losses = [], []

        for epoch in range(1, epochs + 1):
            # ---- Train ----
            model.train()
            running = 0.0
            n_obs = 0

            for batch in train_loader_small:
                x_img, x_cond, y = to_device(batch, device)

                optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=cfg.amp):
                    preds = model(x_img, x_cond).squeeze(-1)
                    loss = loss_fn(preds, y.view_as(preds).float())

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                bs = y.size(0)
                running += loss.item() * bs
                n_obs += bs

            train_epoch_loss = running / max(1, n_obs)
            train_losses.append(train_epoch_loss)

            # ---- Validate ----
            model.eval()
            v_running = 0.0
            v_obs = 0
            with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.amp):
                for batch in val_loader_small:
                    x_img, x_cond, y = to_device(batch, device)
                    preds = model(x_img, x_cond).squeeze(-1)
                    v_loss = loss_fn(preds, y.view_as(preds).float())
                    bs = y.size(0)
                    v_running += v_loss.item() * bs
                    v_obs += bs

            val_epoch_loss = v_running / max(1, v_obs)
            val_losses.append(val_epoch_loss)

            # Log to W&B
            wandb.log({
                "epoch": epoch,
                "train/mse": train_epoch_loss,
                "val/mse": val_epoch_loss,
                "lr": optimizer.param_groups[0]["lr"],
            })

        # Final plot
        plot_and_log_curves(train_losses, val_losses, out_path="loss_curve.png")

        # Also log the final metrics explicitly
        wandb.log({
            "final/train_mse": train_losses[-1],
            "final/val_mse": val_losses[-1],
        })


# --------------------------
# Sweep configuration
# --------------------------
SWEEP_CONFIG = {
    "method": "grid",
    "metric": {"name": "val/mse", "goal": "minimize"},
    "parameters": {
        # ----- Model depth/width -----
        "pre_size":   {"values": ["small", "mid", "big"]},
        "post_size":  {"values": ["small", "mid", "big"]},
        "head":       {"values": ["none", "mid", "wide"]},

        # ----- Activations / Norm / Dropout -----
        "act":            {"values": ["relu", "gelu", "silu"]},
        "norm_groups":    {"values": [4, 8, 16]},
        "conv_dropout":   {"values": [0.0, 0.22, 0.35]},
        "fc_dropout":     {"values": [0.0, 0.22, 0.50]},
        "conv_drop_every":{"values": [1, 2, 3]},

        # ----- Convolution & Pooling geometry -----
        "kernel_size": {"values": [1, 3, 5]},  # padding is set = k//2 in builder
        "pool_kernel": {"values": [2, 2, 3]},  # middle duplicates 2 to keep grid 3-wide but realistic

        # ----- Optim/training -----
        "optimizer":   {"values": ["adam", "adamw", "sgd"]},
        "lr":          {"values": [1e-4, 3e-4, 1e-3]},
        "weight_decay":{"values": [0.0, 1e-5, 1e-4]},
        "batch_size":  {"values": [32, 64, 128]},
        "epochs":      {"values": [20, 20, 20]},  # fixed at 20 but keeps the 3-value pattern
        "amp":         {"values": [True, True, True]},  # keep AMP on in all runs

        # ----- Data fractions (small subsets for speed) -----
        "train_frac":  {"values": [0.05, 0.1, 0.2]},
        "val_frac":    {"values": [0.05, 0.1, 0.2]},

        # ----- Misc -----
        "seed":        {"values": [1337, 2024, 31415]},
    },
}



In [10]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

# import your ModularCondCNN and train_loader here
# from your_module import ModularCondCNN
# from your_data_module import train_loader

def make_subset_loaders(train_loader, train_frac=0.1, val_frac=0.1, seed=1337):
    """Make smaller subsets for fast testing."""
    ds = train_loader.dataset
    n = len(ds)
    n_train = max(1, int(n * train_frac))
    n_val = max(1, int(n * val_frac))
    g = torch.Generator().manual_seed(seed)
    subsets = random_split(ds, [n_train, n_val, n - n_train - n_val], generator=g)
    ds_train, ds_val = subsets[0], subsets[1]

    train_loader_small = DataLoader(
        ds_train, batch_size=train_loader.batch_size, shuffle=True,
        num_workers=train_loader.num_workers, pin_memory=True
    )
    val_loader_small = DataLoader(
        ds_val, batch_size=train_loader.batch_size, shuffle=False,
        num_workers=train_loader.num_workers, pin_memory=True
    )
    return train_loader_small, val_loader_small


def to_device(batch, device):
    x_img, x_cond, y = batch
    return x_img.to(device), x_cond.to(device), y.to(device)


def train_one_run_local():
    """Run one full training session locally without W&B."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # ---- Create small subset ----
    train_loader_small, val_loader_small = make_subset_loaders(train_loader, 0.05, 0.05)

    # ---- Define a basic model (like your mid config) ----
    model = ModularCondCNN(
        in_channels=1,
        cond_dim=4,
        pre_concat_blocks=[(64, True), (128, True), (256, False)],
        post_concat_blocks=[256, 256, 256, 256, 256],
        head_dims=[512, 128],
        act_factory=torch.nn.GELU,
        conv_dropout_p=0.22,
        fc_dropout_p=0.22,
        conv_drop_every=2,
        norm_groups_preferred=8,
        kernel_size=3,
        padding=1,
        pool_kernel=2,
        pool_stride=2,
    ).to(device)

    # ---- Optimizer and loss ----
    optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
    loss_fn = nn.MSELoss()

    epochs = 5  # quick test; you can set 20 later
    scaler = torch.cuda.amp.GradScaler(enabled=True)

    print("Starting training...\n")
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        n_obs = 0
        t_epoch = time.time()

        for batch in train_loader_small:
            x_img, x_cond, y = to_device(batch, device)
            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=True):
                preds = model(x_img, x_cond).squeeze(-1)
                loss = loss_fn(preds, y.view_as(preds).float())

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * y.size(0)
            n_obs += y.size(0)

        train_loss = running_loss / n_obs

        # ---- Validation ----
        model.eval()
        val_loss = 0.0
        n_val = 0
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=True):
            for batch in val_loader_small:
                x_img, x_cond, y = to_device(batch, device)
                preds = model(x_img, x_cond).squeeze(-1)
                l = loss_fn(preds, y.view_as(preds).float())
                val_loss += l.item() * y.size(0)
                n_val += y.size(0)
        val_loss /= n_val

        elapsed = time.time() - t_epoch
        print(f"Epoch {epoch:2d} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | time={elapsed:.2f}s")

    total_time = time.time() - t0
    print(f"\nTotal training time: {total_time:.2f} seconds for {epochs} epochs.")
    print("Done.")

if __name__ == "__main__":
    train_one_run_local()


Using device: cuda


/tmp/ipykernel_181640/2206149470.py:68: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=True)


Starting training...



/tmp/ipykernel_181640/2206149470.py:82: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True):
/tmp/ipykernel_181640/2206149470.py:99: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=True):


KeyboardInterrupt: 

In [11]:
import math, time, torch, torch.nn as nn



use_amp = torch.cuda.is_available()
device = torch.device("cuda" if use_amp else "cpu")
train_loader_small, val_loader_small = make_subset_loaders(train_loader, 0.05, 0.05)

# ---- Define a basic model (like your mid config) ----
model = ModularCondCNN(
    in_channels=1,
    cond_dim=4,
    pre_concat_blocks=[(64, True), (128, True), (256, False)],
    post_concat_blocks=[256, 256, 256, 256, 256],
    head_dims=[512, 128],
    act_factory=torch.nn.GELU,
    conv_dropout_p=0.22,
    fc_dropout_p=0.22,
    conv_drop_every=2,
    norm_groups_preferred=8,
    kernel_size=3,
    padding=1,
    pool_kernel=2,
    pool_stride=2,
).to(device)

model.to(device).eval()
loss_fn = nn.MSELoss()

# pull one batch from your warmed train loader
batch = next(iter(train_loader_small))
x_img, x_cond, y = (t.to(device) for t in batch)

# time K steps and average
K = 10
torch.cuda.synchronize() if use_amp else None
t0 = time.time()
for _ in range(K):
    with torch.amp.autocast("cuda", enabled=use_amp):
        preds = model(x_img, x_cond).squeeze(-1)
        loss  = loss_fn(preds, y.view_as(preds).float())
    if model.training:  # for completeness, but you can time train too
        loss.backward(); optimizer.step(); optimizer.zero_grad()
torch.cuda.synchronize() if use_amp else None
dt = (time.time() - t0) / K

steps_per_epoch = math.ceil(len(train_loader_small.dataset) / train_loader_small.batch_size)
print(f"time_per_step ≈ {dt:.3f}s | steps/epoch ≈ {steps_per_epoch} | "
      f"epoch ETA ≈ {dt * steps_per_epoch:.1f}s")


time_per_step ≈ 0.045s | steps/epoch ≈ 109 | epoch ETA ≈ 4.9s


In [ ]:

# --------------------------
# Entrypoint
# --------------------------
if __name__ == "__main__":
    """
    Before running:
      - Ensure `train_loader` is defined/imported in this namespace.
      - `wandb` must be logged in (e.g., `wandb login` in your shell or env var).
    """
    # Provide your W&B project/entity here:
    WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "modular-condcnn-sweeps")
    WANDB_ENTITY  = os.environ.get("WANDB_ENTITY", None)  # or your username/team

    sweep_id = wandb.sweep(SWEEP_CONFIG, project=WANDB_PROJECT, entity=WANDB_ENTITY)

    # NOTE: A full grid here is very large (3^N). Set a cap on runs for practicality.
    # Increase/decrease `count` depending on how wide you want to explore right now.
    wandb.agent(sweep_id, function=train_one_run, count=60)


wandb: Network error (ReadTimeout), entering retry loop.


KeyboardInterrupt: 